### ANN/MLP building using pytorch

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn 
import torch.optim as optim 
import matplotlib.pyplot as plt 

In [ ]:
torch.__version__

In [ ]:
torch.manual_seed(42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

In [ ]:
df_train = pd.read_csv(r'fashion mnist/fashion-mnist_train.csv')
df_test = pd.read_csv(r'fashion mnist/fashion-mnist_test.csv')

In [ ]:
df_train

In [ ]:
fig, axes = plt.subplots(4,4, figsize=(10,10))
fig.suptitle("First 16 Images", fontsize = 16)

for i, ax in enumerate(axes.flat):
    img = df_train.iloc[i, 1:].values.reshape(28, 28)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"Label: {df_train.iloc[i,0]}")

plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

In [ ]:
X_train = df_train.drop(columns=['label'])
X_test = df_test.drop(columns='label')

y_train = df_train['label']
y_test = df_test['label']

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
y_train

In [ ]:
y_test

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
# create custom dataset class

class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features.to_numpy(), dtype=torch.float32)
        self.labels = torch.tensor(labels.to_numpy(), dtype=torch.long)
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]
        

In [ ]:
train_dataset = CustomDataset(X_train, y_train)


In [ ]:
len(train_dataset)

In [ ]:
train_dataset[0]

In [ ]:
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
# Dataloader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False, pin_memory=True)

In [ ]:
## define nn

class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)
            # softmax in internally defined in the final layer by pytorch
        )
        
    def forward(self, x):
        return self.model(x)

In [ ]:
model = MyNN(X_train.shape[1])
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)


In [ ]:
epochs = 100

for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        
        # move data to gpu
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        
        # forward pass
        outputs = model(batch_features)
        # calculate loss 
        loss = criterion(outputs, batch_labels)
        # back pass
        optimizer.zero_grad()
        loss.backward()
        # update grad
        optimizer.step()
        total_epoch_loss = total_epoch_loss + loss.item()
        
    avg_loss = total_epoch_loss / len(train_loader)    
    print(f"Epoch: {epoch+1}, loss: {avg_loss}")

In [ ]:
# set the model to eval mode
model.eval()

In [ ]:
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        
        # move data to gpu
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        
        outputs = model(batch_features)
        
        _, predicted = torch.max(outputs,1)
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()
    print(correct/total)